[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/cadquery_basics/blob/master/docs/tuto_colab/cadquery_trusses_gc.ipynb)

# Treliças
## CadQuery para Arquitetos e Engenheiros
### Versão Google Colab

---

Neste notebook vamos modelar uma **treliça plana** de forma paramétrica. O processo começa com a criação e visualização de pontos e polilinhas, passa pela definição dos banzos (superior e inferior), e culmina na geração dos nós de conexão e dos elementos (barras).

Todas as medidas estão em **metros**.

---

## Instalação

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("Running in Colab, installing packages...")
    !pip install cadquery cadquery-simpleviewer
else:
    print("Not running in Colab, skipping package installation.")

## Importações

In [ ]:
import cadquery as cq
from cadquery_simpleviewer import show
from copy import copy

---

## Pontos no Espaço 3D

No CadQuery, pontos no espaço são representados por objetos `cq.Vector`. Cada vetor armazena três coordenadas: `x`, `y` e `z`.

A função `show()` da cadquery-simpleViewer exibe `cq.Vector` diretamente como marcadores no visualizador 3D — sem necessidade de criar sólidos ou traces Plotly manualmente.

Vamos criar quatro pontos que esboçam o perfil de uma treliça de cobertura:

In [ ]:
# Quatro pontos definindo um perfil simples
p1 = cq.Vector( 0, 0, 0)
p2 = cq.Vector( 3, 0, 2)
p3 = cq.Vector( 7, 0, 2)
p4 = cq.Vector(10, 0, 0)

show(
    [p1, p2, p3, p4],
    names=["P1", "P2", "P3", "P4"],
    points_display=dict(size=10, color="steelblue", symbol="circle")
)

---

## Polilinha

Uma **polilinha** é uma sequência de segmentos de linha conectando pontos em ordem. Em CadQuery, um conjunto de segmentos conectados é representado por um `cq.Wire`.

O método `cq.Wire.makePolygon()` recebe uma lista de `cq.Vector` e cria um wire com segmentos retos entre eles — exatamente uma polilinha. A cadquery-simpleViewer exibe `cq.Wire` diretamente como linha, com o parâmetro `lines_display` controlando a aparência.

Podemos exibir a polilinha e os pontos nomeados na mesma chamada `show()`:

In [ ]:
# Criando a polilinha como um wire CadQuery
polilinha = cq.Wire.makePolygon([p1, p2, p3, p4])

# Exibindo a polilinha e os nós na mesma cena
show(
    [polilinha, p1, p2, p3, p4],
    names=["Polilinha", "P1", "P2", "P3", "P4"],
    lines_display=dict(color="steelblue", width=4),
    points_display=dict(size=8, color="indianred", symbol="circle")
)

---

## Treliça — Definição dos Banzos

A treliça plana é composta por dois elementos lineares principais chamados **banzos**:

- **Banzo superior** (`banzo_sup`): o elemento de cima, que pode ser reto ou arqueado
- **Banzo inferior** (`banzo_inf`): o elemento de baixo, paralelo ao superior

No CadQuery, usamos `cq.Edge` para criar elementos lineares (retos ou curvos):

| Método | Descrição |
|--------|-----------|
| `cq.Edge.makeLine(v1, v2)` | Segmento de linha reto entre dois pontos |
| `cq.Edge.makeThreePointArc(v1, v2, v3)` | Arco circular passando por três pontos |

### Parâmetros da treliça

In [ ]:
# Parâmetros gerais da treliça
comprimento  = 10.0   # comprimento total em metros
z_sup        = 12.0   # cota do banzo superior (m)
altura_treli = 1.0    # distância vertical entre banzos (m)
n_paineis    = 6      # número de painéis (divisões) da treliça

### Banzo Superior

O banzo superior pode ser definido como uma **linha reta** ou como um **arco** no plano XZ. As duas opções estão no código abaixo — comente uma e descomente a outra para alternar entre os dois perfis:

- **Linha reta**: adequada para treliças com cobertura plana ou levemente inclinada
- **Arco**: adequada para treliças com cobertura curva (treliça Pratt, Howe, etc.)

In [ ]:
# --- Opção 1: Banzo superior reto ---
banzo_sup = cq.Edge.makeLine(
    cq.Vector(0,          0, z_sup),
    cq.Vector(comprimento, 0, z_sup)
)

# --- Opção 2: Banzo superior em arco ---
# Descomente as linhas abaixo e comente a Opção 1 para usar o arco.
# O ponto do meio define a flecha (altura) do arco.
# flecha = 1.5
# banzo_sup = cq.Edge.makeThreePointArc(
#     cq.Vector(0,               0, z_sup),
#     cq.Vector(comprimento / 2, 0, z_sup + flecha),
#     cq.Vector(comprimento,     0, z_sup)
# )

### Banzo Inferior

O banzo inferior é uma cópia do banzo superior deslocada `altura_treli` metros para baixo no eixo Z.

In [ ]:
# Cópia do banzo superior deslocada para baixo
banzo_inf = copy(banzo_sup)
banzo_inf = banzo_inf.translate(cq.Vector(0, 0, -altura_treli))

---

## Geração dos Nós

Os **nós** da treliça são os pontos de conexão entre as barras. Eles são gerados dividindo os banzos em `n_paineis` espaços iguais usando o método `.positionAt(t)`, que retorna um `cq.Vector` para qualquer parâmetro `t` entre `0.0` e `1.0` ao longo do elemento.

Isso funciona tanto para elementos retos quanto para arcos — o método sempre percorre a geometria real do elemento.

In [ ]:
def gerar_nos(aresta, n):
    """
    Divide uma aresta em n espaços iguais e retorna uma lista de cq.Vector.

    aresta : cq.Edge — aresta a dividir
    n      : int     — número de painéis (divisões)
    """
    nos = []
    for i in range(n + 1):
        t = i / n
        nos.append(aresta.positionAt(t))
    return nos


# Nós do banzo superior e inferior como listas de cq.Vector
points_sup = gerar_nos(banzo_sup, n_paineis)
points_inf = gerar_nos(banzo_inf, n_paineis)

print(f"Nós gerados no banzo superior: {len(points_sup)}")
print(f"Nós gerados no banzo inferior: {len(points_inf)}")

print("\nBanzo superior — coordenadas dos nós:")
for i in range(len(points_sup)):
    p = points_sup[i]
    print(f"  [{i}]  x={p.x:.3f}  y={p.y:.3f}  z={p.z:.3f}")

---

## Visualização dos Banzos e Nós

Passamos os dois banzos (`cq.Edge`) e as duas listas de nós (`cq.Vector`) diretamente para `show()`. O parâmetro `lines_display` controla a aparência das arestas e `points_display` controla os marcadores.

Como os dois banzos têm cores diferentes e os dois conjuntos de nós também, passamos `colors` para os banzos e usamos `points_display` para os nós — neste caso com a mesma cor para todos os pontos. Para cores individuais por ponto seria necessário chamar `show()` separadamente para cada grupo.

In [ ]:
# Construindo a lista de objetos e nomes
objetos = [banzo_sup, banzo_inf]
nomes   = ["Banzo superior", "Banzo inferior"]

for i in range(len(points_sup)):
    objetos.append(points_sup[i])
    nomes.append("NS" + str(i))

for i in range(len(points_inf)):
    objetos.append(points_inf[i])
    nomes.append("NI" + str(i))

show(
    objetos,
    names=nomes,
    lines_display=dict(color="steelblue", width=4, samples=60),
    points_display=dict(size=8, color="indianred", symbol="circle")
)

> 💡 **Observe**: os banzos são exibidos como linhas e os nós como marcadores — tudo com uma única chamada `show()`, sem nenhum código Plotly manual.

> ⚠️ **Nota sobre `lines_display` e `points_display`**: os estilos se aplicam uniformemente a todos os elementos do mesmo tipo na chamada. Os dois banzos compartilham a mesma cor de linha, e todos os nós compartilham a mesma cor de marcador. Para estilos diferentes por grupo, use chamadas `show()` separadas.

---

## Geração das Montantes e Diagonais

Os **montantes** são as barras verticais que conectam nós do banzo superior ao nó diretamente abaixo no banzo inferior. As **diagonais** conectam nós alternados criando os triângulos da treliça.

Cada barra é criada com `cq.Edge.makeLine()` a partir de dois `cq.Vector`.

In [ ]:
# --- Montantes: conectam nó superior[i] ao nó inferior[i] ---
montantes = []
for i in range(len(points_sup)):
    barra = cq.Edge.makeLine(points_sup[i], points_inf[i])
    montantes.append(barra)

# --- Diagonais: conectam nó superior[i] ao nó inferior[i+1] ---
diagonais = []
for i in range(len(points_sup) - 1):
    barra = cq.Edge.makeLine(points_sup[i], points_inf[i + 1])
    diagonais.append(barra)

print(f"Montantes gerados : {len(montantes)}")
print(f"Diagonais geradas : {len(diagonais)}")

---

## Visualização da Treliça Completa

Montamos a cena com todos os elementos: banzos, montantes, diagonais e nós.

In [ ]:
# Construindo a lista completa de objetos
objetos = []
nomes   = []

# Banzos
objetos.append(banzo_sup)
nomes.append("Banzo superior")

objetos.append(banzo_inf)
nomes.append("Banzo inferior")

# Montantes
for i in range(len(montantes)):
    objetos.append(montantes[i])
    nomes.append("M" + str(i))

# Diagonais
for i in range(len(diagonais)):
    objetos.append(diagonais[i])
    nomes.append("D" + str(i))

# Nós
for i in range(len(points_sup)):
    objetos.append(points_sup[i])
    nomes.append("NS" + str(i))

for i in range(len(points_inf)):
    objetos.append(points_inf[i])
    nomes.append("NI" + str(i))

show(
    objetos,
    names=nomes,
    lines_display=dict(color="steelblue", width=3, samples=30),
    points_display=dict(size=7, color="indianred", symbol="circle")
)

---

## Resumo

Neste notebook você aprendeu:

- Como representar pontos no espaço com `cq.Vector`
- Como criar polilinhas com `cq.Wire.makePolygon()`
- Como criar arestas retas e arcos com `cq.Edge.makeLine()` e `cq.Edge.makeThreePointArc()`
- Como dividir uma aresta em nós equidistantes com `.positionAt(t)`
- Como exibir `cq.Edge`, `cq.Wire` e `cq.Vector` diretamente com `show()` — sem código Plotly manual
- Como usar `lines_display` e `points_display` para controlar a aparência de linhas e marcadores
- Como montar uma treliça paramétrica com banzos, montantes e diagonais

---
*CadQuery para Arquitetos — Notebook Treliças (versão Google Colab)*